# Auditoría de Resultados y Órdenes de Compra — agosto 2026

**Solo lectura · clon Google Sheets · sin ZIP ni API ChileCompra.**

Compara los procesos marcados como seleccionados en `Resultados` (65.945) con las órdenes de compra históricas de `OrdenesCompra` (153.220). La existencia de una OC relacionada por código de proceso y proveedor **no equivale por sí sola a una adjudicación oficial verificada**. Los montos sirven de control adicional; una OC puede incluir partidas o ajustes. Solo produce informes CSV/JSON locales, sin escribir en Sheets.

[Repositorio](https://github.com/juliopez/MercadoPublico-Inteligencia)

In [ ]:
!pip -q install gspread google-auth
from google.colab import auth,files
from google.auth import default
import gspread, csv,io,unicodedata,re,json,collections,datetime,decimal,os
auth.authenticate_user()
cred,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
gc=gspread.authorize(cred)
CLON='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
libro=gc.open_by_key(CLON)
cab_r=['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc']
cab_oc=['codigo_oc','codigo_proceso','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto']
print('Conectado al clon en modo lectura')

## Extracción por páginas

Se leen exclusivamente las columnas necesarias y se respetan las cabeceras. La lectura por bloques permite supervisar progreso y detectar límites de Google Sheets. No se consulta el SQLite ni se abre ZIP.

In [ ]:
def extraer(titulo,cab,block=5000):
 ws=libro.worksheet(titulo)
 assert ws.row_values(1)==cab, f'Cabecera inesperada en {titulo}'
 lim=ws.row_count
 datos=[]
 for start in range(2,lim+1,block):
  end=min(lim,start+block-1)
  rows=ws.get(f'A{start}:J{end}',value_render_option='UNFORMATTED_VALUE')
  for row in rows:
   if not row or not any(str(v).strip() for v in row):continue
   row=[str(v).strip() if v is not None else '' for v in row]
   row+=['']*(len(cab)-len(row))
   datos.append(dict(zip(cab,row[:len(cab)])))
  if (start-2)//block%5==0:print(titulo,'filas recuperadas:',len(datos))
 print(titulo,'filas recuperadas finales:',len(datos))
 return datos
resultados=extraer('Resultados',cab_r)
ordenes=extraer('OrdenesCompra',cab_oc)
assert resultados and ordenes,'No hay suficientes datos para auditar'
print('Datos listos, sin escritura')

## Cruce conservador

Normalizar RUT no es posible: la pestaña no contiene RUT de proveedor. Se usan coincidencias por nombre empresarial tras quitar tildes, puntuación y espacios; no se asume equivalencia entre razones sociales distintas. El cruce principal exige código de proceso idéntico. Se reportan separadamente proveedores iguales, distintos y montos idénticos, así como códigos OC duplicados y procesos con múltiples órdenes.

In [ ]:
def normal(t):
 t=unicodedata.normalize('NFKD',str(t or '').upper())
 t=''.join(x for x in t if not unicodedata.combining(x))
 return re.sub(r'[^A-Z0-9]+','',t)
def monto(v):
 s=str(v or '').strip()
 if not s:return None
 try:return decimal.Decimal(s)
 except decimal.InvalidOperation:return None
por_codigo=collections.defaultdict(list)
codigos_oc=collections.Counter()
for oc in ordenes:
 codigo=oc['codigo_proceso'].upper().strip()
 if codigo:por_codigo[codigo].append(oc)
 if oc['codigo_oc']:codigos_oc[oc['codigo_oc']]+=1
duplicados_oc={k:v for k,v in codigos_oc.items() if v>1}
duplicados_resultados={k:v for k,v in collections.Counter(r['codigo'].upper() for r in resultados).items() if v>1}
filas=[];cuenta=collections.Counter()
for r in resultados:
 codigo=r['codigo'].upper().strip()
 vinculadas=por_codigo.get(codigo,[])
 nm=normal(r['adjudicatario']);mo=monto(r['monto_oferta_seleccionada_clp'])
 iguales=[o for o in vinculadas if nm and normal(o['proveedor'])==nm]
 montos=[o for o in iguales if mo is not None and monto(o['monto_total_oc_clp'])==mo]
 if not vinculadas:estado='SIN_OC_ASOCIADA'
 elif not iguales:estado='OC_ASOCIADA_PROVEEDOR_DISTINTO'
 elif montos:estado='CODIGO_PROVEEDOR_MONTO_COINCIDEN'
 else:estado='CODIGO_PROVEEDOR_COINCIDEN_MONTO_DISTINTO'
 cuenta[estado]+=1
 filas.append({'codigo_proceso':codigo,'proveedor_seleccionado':r['adjudicatario'],
  'monto_oferta_seleccionada_clp':r['monto_oferta_seleccionada_clp'],
  'estado_auditoria':estado,'numero_oc_codigo':len(vinculadas),
  'numero_oc_mismo_proveedor':len(iguales),'numero_oc_mismo_monto':len(montos),
  'codigos_oc_coincidentes':' | '.join(o['codigo_oc'] for o in (montos if montos else iguales)[:10]),
  'montos_oc_relacionadas':' | '.join(o['monto_total_oc_clp'] for o in iguales[:10]),
  'identificador_oferta_sintetico':r['id_cotizacion_ganadora'],
  'validado_original':r['validado']})
codigos_resultado={r['codigo'].upper().strip() for r in resultados}
oc_sin_resultado=sum(1 for o in ordenes if o['codigo_proceso'].upper().strip() not in codigos_resultado)
informe={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'clon':'auditoria solo lectura',
 'filas_resultados':len(resultados),'filas_ordenes_compra':len(ordenes),
 'resultados_por_categoria':dict(cuenta),
 'resultados_codigo_duplicado':len(duplicados_resultados),
 'oc_codigo_duplicado':len(duplicados_oc),
 'ordenes_sin_resultado_en_pestana':oc_sin_resultado,
 'alcance':'Coincidencia documental indicativa, NO adjudicacion oficialmente verificada',
 'observacion':'La comparación de razones sociales normalizadas no reemplaza cotejo por RUT oficial'}
print(json.dumps(informe,ensure_ascii=False,indent=2))

## Exportar informe

Descarga el resumen JSON y el detalle completo CSV para revisión. El notebook **no cambia** `Resultados`, `OrdenesCompra`, producción ni el frontend.

In [ ]:
raiz='/content/auditoria_resultados_oc_agosto_2026'
with open(raiz+'_resumen.json','w',encoding='utf-8') as f:json.dump(informe,f,ensure_ascii=False,indent=2)
with open(raiz+'_detalle.csv','w',encoding='utf-8-sig',newline='') as f:
 w=csv.DictWriter(f,fieldnames=list(filas[0]))
 w.writeheader();w.writerows(filas)
print('Generados:',raiz+'_resumen.json',raiz+'_detalle.csv')
files.download(raiz+'_resumen.json')
files.download(raiz+'_detalle.csv')